# Pure Mathematical Integrated Deep Learning Model (From First Principles)

This notebook formulates and implements a complete Deep Learning model purely from **multivariable matrix calculus**, **information theory**, **adaptive dynamical optimization**, and **continuous-time differential/integral equations (Neural ODEs)** using only raw linear algebra (`NumPy`).

---

## Mathematical Architecture Overview

Let $\mathcal{D} = \{(\mathbf{x}_i, \mathbf{y}_i)\}_{i=1}^N$ be a dataset where inputs reside in $\mathbf{X} \in \mathbb{R}^{B \times d_0}$ and one-hot targets reside on the probability simplex $\mathbf{Y} \in \Delta^{C-1} \subset \mathbb{R}^{B \times C}$.

An $L$-layer deep network with Batch Normalization and Leaky-ReLU activations is a parameterized composition of operators $\mathcal{F}_{\Theta} : \mathbb{R}^{B \times d_0} \to \Delta^{C-1}$:

$$\mathcal{F}_{\Theta}(\mathbf{X}) = \left( \boldsymbol{\sigma}_{\text{softmax}} \circ \mathcal{A}^{[L]} \circ \Phi^{[L-1]} \circ \mathcal{B}^{[L-1]} \circ \mathcal{A}^{[L-1]} \circ \cdots \circ \Phi^{[1]} \circ \mathcal{B}^{[1]} \circ \mathcal{A}^{[1]} \right)(\mathbf{X})$$

In [3]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
print('NumPy version:', np.__version__)

ModuleNotFoundError: No module named 'numpy'

## 1. Synthetic Non-Linear Manifold ($C$-Class Archimedes Spiral)

We construct a non-linearly separable manifold in $\mathbb{R}^2$ parameterized by radius $r \in [0.05, 1.0]$ and angular phase $\theta_c(r) = 4.2 r + c \cdot 4.2 + \eta$, where $\eta \sim \mathcal{N}(0, \sigma^2)$:

$$\mathbf{x}_i = \begin{bmatrix} r_i \sin(\theta_c(r_i)) & r_i \cos(\theta_c(r_i)) \end{bmatrix} \in \mathbb{R}^{1 \times 2}$$

In [ ]:
def generate_spiral_manifold(n_per_class=200, n_classes=3, noise=0.18):
    N = n_per_class * n_classes
    X = np.zeros((N, 2), dtype=np.float64)
    y_idx = np.zeros(N, dtype=int)
    
    for c in range(n_classes):
        ix = range(n_per_class * c, n_per_class * (c + 1))
        r = np.linspace(0.05, 1.0, n_per_class)
        theta = np.linspace(c * 4.2, (c + 1) * 4.2, n_per_class) + np.random.randn(n_per_class) * noise
        X[ix] = np.c_[r * np.sin(theta), r * np.cos(theta)]
        y_idx[ix] = c
        
    # One-hot encode onto the probability simplex Delta^{C-1}
    Y = np.eye(n_classes, dtype=np.float64)[y_idx]
    return X, Y, y_idx

X_train, Y_train, y_train_idx = generate_spiral_manifold(n_per_class=200, n_classes=3)
plt.figure(figsize=(5, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train_idx, cmap='viridis', s=20, edgecolors='k', lw=0.4)
plt.title('Non-Linear 3-Class Spiral Manifold in R^2')
plt.xlabel('x_1')
plt.ylabel('x_2')
plt.show()

## 2. Exact Matrix Calculus Engine (Zero Autograd)

### Forward Operators ($l = 1, \dots, L-1$)
1. **Affine Transformation ($\mathcal{A}^{[l]}$):**
   $$\mathbf{Z}^{[l]} = \mathbf{A}^{[l-1]} \mathbf{W}^{[l]} + \mathbf{1}_B (\mathbf{b}^{[l]})^\top$$
2. **Batch Normalization ($\mathcal{B}^{[l]}$):**
   $$\mu_j^{[l]} = \frac{1}{B} \sum_{i=1}^B Z_{ij}^{[l]}, \qquad \left(\sigma_j^{[l]}\right)^2 = \frac{1}{B} \sum_{i=1}^B \left(Z_{ij}^{[l]} - \mu_j^{[l]}\right)^2$$
   $$\widehat{Z}_{ij}^{[l]} = \frac{Z_{ij}^{[l]} - \mu_j^{[l]}}{\sqrt{\left(\sigma_j^{[l]}\right)^2 + \varepsilon}}, \qquad \widetilde{Z}_{ij}^{[l]} = \gamma_j^{[l]} \widehat{Z}_{ij}^{[l]} + \beta_j^{[l]}$$
3. **Leaky-ReLU Nonlinearity ($\Phi^{[l]}$):**
   $$A_{ij}^{[l]} = \max\left(\alpha \widetilde{Z}_{ij}^{[l]},\, \widetilde{Z}_{ij}^{[l]}\right)$$

### Backward Fréchet Adjoint Operators
- **Terminal Softmax + Cross-Entropy Adjoint:**
  $$\mathbf{G}_{\mathbf{Z}}^{[L]} = \frac{\partial \mathcal{L}}{\partial \mathbf{Z}^{[L]}} = \frac{1}{B}\left(\widehat{\mathbf{P}} - \mathbf{Y}\right)$$
- **Exact BatchNorm Total Derivative through $\widehat{\mathbf{Z}}, \boldsymbol{\mu}, \boldsymbol{\sigma}^2$:**
  $$\mathbf{G}_{\mathbf{Z}}^{[l]} = \frac{\boldsymbol{\gamma}^{[l]}}{B \sqrt{(\boldsymbol{\sigma}^{[l]})^2 + \varepsilon}} \odot \left( B \mathbf{G}_{\widetilde{\mathbf{Z}}}^{[l]} - \mathbf{1}_B \sum_{k=1}^B \left(\mathbf{G}_{\widetilde{\mathbf{Z}}}^{[l]}\right)_{k,:} - \widehat{\mathbf{Z}}^{[l]} \odot \mathbf{1}_B \sum_{k=1}^B \left(\mathbf{G}_{\widetilde{\mathbf{Z}}}^{[l]} \odot \widehat{\mathbf{Z}}^{[l]}\right)_{k,:} \right)$$

In [ ]:
class PureMathDeepNetwork:
    def __init__(self, layer_dims, alpha=0.01, reg_lambda=1e-4, eps=1e-8):
        self.L = len(layer_dims) - 1
        self.alpha = alpha
        self.reg_lambda = reg_lambda
        self.eps = eps
        self.params = {}
        
        # Spectral Variance-Preserving Kaiming-He Initialization
        for l in range(1, self.L + 1):
            d_in, d_out = layer_dims[l - 1], layer_dims[l]
            std = np.sqrt(2.0 / ((1.0 + self.alpha**2) * d_in))
            self.params[f'W{l}'] = np.random.randn(d_in, d_out) * std
            self.params[f'b{l}'] = np.zeros((1, d_out), dtype=np.float64)
            if l < self.L:
                self.params[f'gamma{l}'] = np.ones((1, d_out), dtype=np.float64)
                self.params[f'beta{l}'] = np.zeros((1, d_out), dtype=np.float64)

    def forward(self, X):
        cache = {'A0': X}
        A = X
        for l in range(1, self.L):
            W, b = self.params[f'W{l}'], self.params[f'b{l}']
            gamma, beta = self.params[f'gamma{l}'], self.params[f'beta{l}']
            
            # 1. Affine operator
            Z = A @ W + b
            
            # 2. Batch Normalization operator
            mu = np.mean(Z, axis=0, keepdims=True)
            var = np.var(Z, axis=0, keepdims=True)
            std_inv = 1.0 / np.sqrt(var + self.eps)
            Z_hat = (Z - mu) * std_inv
            Z_tilde = gamma * Z_hat + beta
            
            # 3. Pointwise Leaky-ReLU operator
            A = np.where(Z_tilde > 0, Z_tilde, self.alpha * Z_tilde)
            
            cache.update({
                f'Z{l}': Z, f'mu{l}': mu, f'var{l}': var,
                f'std_inv{l}': std_inv, f'Z_hat{l}': Z_hat,
                f'Z_tilde{l}': Z_tilde, f'A{l}': A
            })
            
        # Terminal Logit & Stabilized Softmax Operator
        ZL = A @ self.params[f'W{self.L}'] + self.params[f'b{self.L}']
        Z_shifted = ZL - np.max(ZL, axis=1, keepdims=True)
        exp_Z = np.exp(Z_shifted)
        P_hat = exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
        
        cache[f'Z{self.L}'] = ZL
        cache['P_hat'] = P_hat
        return P_hat, cache

    def compute_loss(self, X, Y):
        B = X.shape[0]
        P_hat, cache = self.forward(X)
        ce_loss = -np.sum(Y * np.log(np.clip(P_hat, 1e-15, 1.0))) / B
        l2_loss = 0.5 * self.reg_lambda * sum(
            np.sum(self.params[f'W{l}'] ** 2) for l in range(1, self.L + 1)
        )
        return ce_loss + l2_loss, cache

    def backward(self, Y, cache):
        B = Y.shape[0]
        grads = {}
        
        # Terminal Adjoint: dL/dZ^{[L]} = (P_hat - Y) / B
        G_Z = (cache['P_hat'] - Y) / B
        grads[f'W{self.L}'] = cache[f'A{self.L-1}'].T @ G_Z + self.reg_lambda * self.params[f'W{self.L}']
        grads[f'b{self.L}'] = np.sum(G_Z, axis=0, keepdims=True)
        G_A = G_Z @ self.params[f'W{self.L}'].T
        
        # Reverse-Mode Matrix Adjoint Recursion
        for l in range(self.L - 1, 0, -1):
            Z_tilde = cache[f'Z_tilde{l}']
            Z_hat = cache[f'Z_hat{l}']
            std_inv = cache[f'std_inv{l}']
            gamma = self.params[f'gamma{l}']
            
            # Leaky-ReLU Jacobian product
            dPhi = np.where(Z_tilde > 0, 1.0, self.alpha)
            G_Z_tilde = G_A * dPhi
            
            # BatchNorm scale & shift gradients
            grads[f'gamma{l}'] = np.sum(G_Z_tilde * Z_hat, axis=0, keepdims=True)
            grads[f'beta{l}'] = np.sum(G_Z_tilde, axis=0, keepdims=True)
            
            # Exact BatchNorm Total Derivative through mu and sigma^2
            G_Z = (gamma * std_inv / B) * (
                B * G_Z_tilde
                - np.sum(G_Z_tilde, axis=0, keepdims=True)
                - Z_hat * np.sum(G_Z_tilde * Z_hat, axis=0, keepdims=True)
            )
            
            # Affine parameter & input gradients
            grads[f'W{l}'] = cache[f'A{l-1}'].T @ G_Z + self.reg_lambda * self.params[f'W{l}']
            grads[f'b{l}'] = np.sum(G_Z, axis=0, keepdims=True)
            if l > 1:
                G_A = G_Z @ self.params[f'W{l}'].T
                
        return grads

## 3. Mathematical Proof of Correctness: Taylor-Series Finite-Difference Verification

By Taylor's theorem, the centered difference operator approximates the exact directional derivative to $O(h^2)$:

$$\frac{\partial \mathcal{L}(\Theta)}{\partial \theta_k} = \frac{\mathcal{L}(\Theta + h \mathbf{e}_k) - \mathcal{L}(\Theta - h \mathbf{e}_k)}{2h} + O(h^2)$$

Below we verify that the relative Euclidean error between our analytical matrix calculus $\nabla_a \mathcal{L}$ and numerical calculus $\nabla_n \mathcal{L}$ is $\le 10^{-9}$ across all weights, biases, and BatchNorm parameters.

In [ ]:
def verify_analytical_gradients():
    net = PureMathDeepNetwork(layer_dims=[2, 8, 8, 3], alpha=0.05, reg_lambda=1e-3)
    X_sub, Y_sub = X_train[:16], Y_train[:16]
    _, cache = net.compute_loss(X_sub, Y_sub)
    analytic_grads = net.backward(Y_sub, cache)
    
    h = 1e-6
    print(f"{'Parameter':<10} | {'Rel. L2 Error ||g_a - g_n|| / ||g_a + g_n||':<45}")
    print('-' * 60)
    for name, param in net.params.items():
        num_grad = np.zeros_like(param)
        it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
        while not it.finished:
            idx = it.multi_index
            orig = param[idx]
            
            param[idx] = orig + h
            loss_plus, _ = net.compute_loss(X_sub, Y_sub)
            
            param[idx] = orig - h
            loss_minus, _ = net.compute_loss(X_sub, Y_sub)
            
            param[idx] = orig
            num_grad[idx] = (loss_plus - loss_minus) / (2.0 * h)
            it.iternext()
            
        g_a = analytic_grads[name]
        rel_error = np.linalg.norm(g_a - num_grad) / (np.linalg.norm(g_a + num_grad) + 1e-15)
        print(f"{name:<10} | {rel_error:.4e}")

verify_analytical_gradients()

## 4. Adam Dynamical System Optimization & Manifold Decision Surface

We integrate the parameter trajectory using bias-corrected exponential moving averages of the first and second moments of the gradient:

$$\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1 - \beta_1)\mathbf{g}_t, \qquad \mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1 - \beta_2)\left(\mathbf{g}_t \odot \mathbf{g}_t\right)$$
$$\widehat{\mathbf{m}}_t = \frac{\mathbf{m}_t}{1 - \beta_1^t}, \qquad \widehat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_2^t}, \qquad \theta_t = \theta_{t-1} - \eta \frac{\widehat{\mathbf{m}}_t}{\sqrt{\widehat{\mathbf{v}}_t} + \epsilon}$$

In [ ]:
net = PureMathDeepNetwork(layer_dims=[2, 64, 64, 3], alpha=0.01, reg_lambda=1e-4)
m_state = {k: np.zeros_like(v) for k, v in net.params.items()}
v_state = {k: np.zeros_like(v) for k, v in net.params.items()}

lr, beta1, beta2, eps_adam = 0.015, 0.9, 0.999, 1e-8
epochs = 300
loss_history = []

for t in range(1, epochs + 1):
    loss, cache = net.compute_loss(X_train, Y_train)
    grads = net.backward(Y_train, cache)
    loss_history.append(loss)
    
    for k in net.params:
        m_state[k] = beta1 * m_state[k] + (1.0 - beta1) * grads[k]
        v_state[k] = beta2 * v_state[k] + (1.0 - beta2) * (grads[k] ** 2)
        m_hat = m_state[k] / (1.0 - beta1 ** t)
        v_hat = v_state[k] / (1.0 - beta2 ** t)
        net.params[k] -= lr * m_hat / (np.sqrt(v_hat) + eps_adam)

P_final, _ = net.forward(X_train)
acc = np.mean(np.argmax(P_final, axis=1) == y_train_idx) * 100
print(f"Final Loss: {loss_history[-1]:.4f} | Training Accuracy: {acc:.2f}%")

# Plot Loss Convergence & Learned Riemannian Decision Manifold
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(loss_history, color='navy', lw=2)
axes[0].set_title('Regularized Cross-Entropy Objective L(Theta)')
axes[0].set_xlabel('Iteration t')
axes[0].set_ylabel('Loss')

xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 150), np.linspace(-1.2, 1.2, 150))
grid = np.c_[xx.ravel(), yy.ravel()]
P_grid, _ = net.forward(grid)
Z_pred = np.argmax(P_grid, axis=1).reshape(xx.shape)

axes[1].contourf(xx, yy, Z_pred, alpha=0.35, cmap='viridis')
axes[1].scatter(X_train[:, 0], X_train[:, 1], c=y_train_idx, cmap='viridis', s=18, edgecolors='k', lw=0.4)
axes[1].set_title(f'Learned Partition of R^2 (Acc: {acc:.1f}%)')
plt.tight_layout()
plt.show()

## 5. Continuous-Time Integrated Neural ODE & Pontryagin Adjoint Solver

Rather than discrete layers, a **Continuous-Depth Neural ODE** defines hidden states via an integral over $t \in [0, T]$:

$$\mathbf{h}(T) = \mathbf{h}(0) + \int_0^T \tanh\left(\mathbf{h}(t)\mathbf{W} + \mathbf{b}\right) \mathrm{d}t$$

We integrate $\mathbf{h}(t)$ forward using 4th-order Runge-Kutta (RK4) and compute exact parameter gradients by integrating the **Pontryagin Adjoint ODE** backward from $t=T$ to $t=0$:

$$\frac{\mathrm{d}\mathbf{a}(t)}{\mathrm{d}t} = -\left(\mathbf{a}(t) \odot \text{sech}^2(\mathbf{h}(t)\mathbf{W} + \mathbf{b})\right)\mathbf{W}^\top, \qquad \nabla_{\mathbf{W}}\mathcal{L} = \int_0^T \mathbf{h}(t)^\top \left(\mathbf{a}(t) \odot \text{sech}^2(\mathbf{h}(t)\mathbf{W} + \mathbf{b})\right) \mathrm{d}t$$

In [ ]:
def ode_dynamics(h, W, b):
    return np.tanh(h @ W + b)

def rk4_forward_integrate(h0, W, b, T=1.0, steps=20):
    dt = T / steps
    traj = [h0]
    h = h0.copy()
    for _ in range(steps):
        k1 = ode_dynamics(h, W, b)
        k2 = ode_dynamics(h + 0.5 * dt * k1, W, b)
        k3 = ode_dynamics(h + 0.5 * dt * k2, W, b)
        k4 = ode_dynamics(h + dt * k3, W, b)
        h = h + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        traj.append(h)
    return np.array(traj)  # Shape: (steps+1, B, d)

def pontryagin_adjoint_backward(traj, a_T, W, b, T=1.0):
    steps = len(traj) - 1
    dt = T / steps
    a = a_T.copy()
    dW = np.zeros_like(W)
    db = np.zeros_like(b)
    
    for s in range(steps, 0, -1):
        h = traj[s]
        sech2 = 1.0 - np.tanh(h @ W + b) ** 2
        local_grad = a * sech2
        
        # Integrate parameter sensitivity dL/dW and dL/db
        dW += (h.T @ local_grad) * dt
        db += np.sum(local_grad, axis=0, keepdims=True) * dt
        
        # Step adjoint state backward: a(t - dt) = a(t) - dt * da/dt
        da_dt = -local_grad @ W.T
        a = a - dt * da_dt
    return a, dW, db

# Demonstrate Continuous-Time Neural ODE Flow on a 2D Vector Field
W_ode = np.array([[-0.4, -1.2], [1.2, -0.4]])
b_ode = np.zeros((1, 2))
h0_samples = X_train[::15]
traj = rk4_forward_integrate(h0_samples, W_ode, b_ode, T=2.0, steps=40)

plt.figure(figsize=(6, 4.5))
for i in range(h0_samples.shape[0]):
    plt.plot(traj[:, i, 0], traj[:, i, 1], alpha=0.6, lw=1.2)
plt.scatter(traj[0, :, 0], traj[0, :, 1], c='red', s=20, label='Initial State h(0)')
plt.scatter(traj[-1, :, 0], traj[-1, :, 1], c='blue', s=20, label='Integrated State h(T)')
plt.title('Continuous-Time Neural ODE Integral Flow h(T) = h(0) + int f(h(t), theta) dt')
plt.legend()
plt.show()